# Praktikum MLOps Minggu 6
## Model Experimentation and Tracking dengan MLflow

Notebook ini dirancang mengikuti komponen Tugas Praktikum pada materi MLOps Minggu 6:
1. Menjalankan MLflow Tracking Server lokal.
2. Menggunakan dataset klasifikasi citra kecil, yaitu Fashion-MNIST.
3. Melatih CNN baseline dan mencatat parameter, metrik per epoch, serta model.
4. Melakukan minimal 20 trial hyperparameter tuning dengan Optuna dan nested runs MLflow.
5. Menjalankan konfigurasi terbaik dengan tiga random seed dan melaporkan mean ± standard deviation.
6. Melakukan ablation study pada dua komponen, yaitu augmentasi dan dropout.
7. Mendaftarkan model terbaik, memberi alias `champion`, lalu memuat dan menguji prediksi melalui alias.
8. Melacak dataset menggunakan DVC dan mencatat identitas/hash dataset pada setiap run.

**Cara kerja:** jalankan sel secara berurutan. Sel yang membutuhkan waktu lebih lama diberi penjelasan. Hasil angka pada laporan harus diambil dari output aktual notebook dan MLflow, bukan diperkirakan.

## 0. Persiapan sebelum menjalankan notebook

Di PowerShell, dari folder proyek `D:\mlops-week6`, pastikan paket utama terpasang:

```powershell
python -m pip install "mlflow>=3" torch torchvision scikit-learn matplotlib pandas optuna dvc
```

Kemudian jalankan server MLflow pada **terminal PowerShell pertama** dan biarkan terminal tersebut tetap terbuka:

```powershell
python -m mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000
```

Buka `http://127.0.0.1:5000` di browser untuk memastikan UI dapat diakses. Jalankan notebook dari folder proyek yang sama agar lokasi basis data, artefak, dan dataset konsisten.

Jika `torch` atau `torchvision` gagal diinstal pada Python 3.13, jangan lanjut dengan environment yang setengah terpasang. Catat pesan error dan sesuaikan versi paket melalui instruksi resmi PyTorch untuk sistem Anda.

## 1. Import library dan konfigurasi eksperimen

In [2]:
import os
import json
import time
import hashlib
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import mlflow
import mlflow.pytorch
import optuna

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
for folder in [DATA_DIR, RAW_DATA_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "fashion-mnist-cnn-mlops-week6"
REGISTERED_MODEL_NAME = "fashion-mnist-cnn"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("MLflow:", mlflow.__version__)
print("Device:", DEVICE)
print("Project directory:", PROJECT_DIR)

Python: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.12.1+cpu
MLflow: 3.17.0
Device: cpu
Project directory: d:\mlops-week6\notebooks


## 2. Reproducibility dan seed

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Determinisme dapat memperlambat pelatihan, tetapi membantu konsistensi.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 3. Unduh Fashion-MNIST dan siapkan pembagian data

Dataset dipisahkan menjadi data pelatihan, validasi, dan test. Validasi digunakan selama pemilihan konfigurasi. Test set ditahan sampai evaluasi final untuk mengurangi risiko pemilihan model berdasarkan test set.

Agar eksperimen tuning dapat dikerjakan dengan waktu yang wajar, notebook menggunakan subset data pelatihan. Angka subset dapat dinaikkan setelah pipeline berjalan lancar.

In [15]:
# Transformasi dasar: tensor float dengan nilai piksel pada rentang [0, 1].
base_transform = transforms.ToTensor()

train_full = datasets.FashionMNIST(
    root=str(RAW_DATA_DIR), train=True, download=True, transform=base_transform
)
test_dataset = datasets.FashionMNIST(
    root=str(RAW_DATA_DIR), train=False, download=True, transform=base_transform
)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Subset untuk iterasi awal. Ubah None jika ingin memakai semua 60.000 data train.
MAX_TRAIN_SAMPLES = 12000
indices = np.arange(len(train_full))
if MAX_TRAIN_SAMPLES is not None:
    indices = indices[:MAX_TRAIN_SAMPLES]

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=np.array(train_full.targets)[indices]
)

train_dataset = Subset(train_full, train_indices.tolist())
val_dataset = Subset(train_full, val_indices.tolist())

print("Jumlah data train:", len(train_dataset))
print("Jumlah data validasi:", len(val_dataset))
print("Jumlah data test:", len(test_dataset))

Jumlah data train: 9600
Jumlah data validasi: 2400
Jumlah data test: 10000


C:\Users\ginda fajar\AppData\Local\Temp\ipykernel_26868\3728557882.py:26: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  stratify=np.array(train_full.targets)[indices]


## 4. Versi dataset dan hash untuk pencatatan run

Sel ini membuat hash berbasis label dan piksel dari subset yang digunakan, sehingga setiap run dapat mencatat identitas data. Hash ini berguna untuk audit eksperimen, tetapi tidak menggantikan DVC. Bagian DVC akan dilakukan pada sel tersendiri.

In [ ]:
def dataset_hash(dataset, max_items=None):
    h = hashlib.sha256()
    n = len(dataset) if max_items is None else min(len(dataset), max_items)
    for i in range(n):
        image, label = dataset[i]
        h.update(image.contiguous().cpu().numpy().tobytes())
        h.update(str(int(label)).encode("utf-8"))
    return h.hexdigest()

# Perhitungan hash penuh pada 12.000 data mungkin memerlukan sedikit waktu.
DATASET_HASH = dataset_hash(train_dataset)
print("Dataset hash (SHA-256):", DATASET_HASH)

dataset_metadata = {
    "dataset": "Fashion-MNIST",
    "train_subset_size": len(train_dataset),
    "validation_size": len(val_dataset),
    "test_size": len(test_dataset),
    "split_seed": 42,
    "hash_algorithm": "SHA-256",
    "train_subset_hash": DATASET_HASH,
}
with open(ARTIFACT_DIR / "dataset_metadata.json", "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, indent=2)

Dataset hash (SHA-256): a52770774d7b55e835496039b885f3a0c6de89b7eaf8bb67d69700dce73a87ca


## 5. DVC: inisialisasi dan pelacakan data

Jalankan bagian ini setelah folder proyek sudah menjadi Git repository. Jika repository belum dibuat, jalankan perintah `git init` dari terminal proyek terlebih dahulu. DVC akan melacak file data mentah yang diunduh oleh torchvision. Folder cache DVC tidak perlu dimasukkan ke Git.

Perintah pada sel ini menginisialisasi DVC bila belum tersedia, lalu mencoba menambahkan folder data Fashion-MNIST ke tracking DVC. Jika DVC menyatakan folder belum dapat ditambahkan atau ada konfigurasi yang perlu disesuaikan, baca output terminal dan perbaiki sebelum lanjut.

In [ ]:
def run_command(args):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=PROJECT_DIR, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("Return code:", result.returncode)
    return result.returncode

git_dir = PROJECT_DIR / ".git"
if not git_dir.exists():
    print("Git repository belum ditemukan. Inisialisasi Git dari terminal proyek dengan: git init")

dvc_dir = PROJECT_DIR / ".dvc"
if not dvc_dir.exists():
    run_command([sys.executable, "-m", "dvc", "init"])

fashion_data_path = RAW_DATA_DIR / "FashionMNIST"
if fashion_data_path.exists() and git_dir.exists():
    run_command([sys.executable, "-m", "dvc", "add", str(fashion_data_path.relative_to(PROJECT_DIR))])
else:
    print("Lewati dvc add: pastikan folder data FashionMNIST dan Git repository tersedia.")

print("Catatan: commit berkas .dvc dan .gitignore yang dihasilkan menggunakan Git setelah memeriksa isinya.")

Git repository belum ditemukan. Inisialisasi Git dari terminal proyek dengan: git init
$ c:\python313\python.exe -m dvc init
ERROR: failed to initiate DVC - d:\mlops-week6\notebooks is not tracked by any supported SCM tool (e.g. Git). Use `--no-scm` if you don't want to use any SCM or `--subdir` if initializing inside a subdirectory of a parent SCM repository.

Return code: 1
Lewati dvc add: pastikan folder data FashionMNIST dan Git repository tersedia.
Catatan: commit berkas .dvc dan .gitignore yang dihasilkan menggunakan Git setelah memeriksa isinya.


## 6. Definisi CNN

Model ini menggunakan dua lapisan konvolusi, ReLU, max pooling, dropout opsional, dan lapisan klasifikasi. Arsitektur dibuat sederhana agar fokus praktikum berada pada experiment tracking dan perbandingan eksperimen.

In [26]:
class FashionCNN(nn.Module):
    def __init__(self, channels=16, dropout=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(channels, channels * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels * 2 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

## 7. Fungsi training dan evaluasi

Metrik dicatat setiap epoch agar kurva pelatihan dapat ditinjau. Augmentasi hanya diterapkan pada data pelatihan; data validasi dan test menggunakan transformasi dasar. Untuk menjaga perbandingan tetap adil, validasi tidak menggunakan augmentasi acak.

In [22]:
def make_train_dataset(use_augmentation=False):
    if not use_augmentation:
        return train_dataset

    augmented_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
    ])
    augmented_full = datasets.FashionMNIST(
        root=str(RAW_DATA_DIR), train=True, download=True, transform=augmented_transform
    )
    # Gunakan indeks split yang sama dengan baseline.
    return Subset(augmented_full, train_indices.tolist())


def make_loaders(batch_size=128, use_augmentation=False, seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)
    tr_ds = make_train_dataset(use_augmentation=use_augmentation)
    train_loader = DataLoader(
        tr_ds, batch_size=batch_size, shuffle=True, num_workers=0,
        generator=generator
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_items = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_items += y.size(0)
    return total_loss / total_items, total_correct / total_items


def train_model(
    params,
    seed=42,
    epochs=3,
    use_augmentation=False,
    use_dropout=True,
    log_each_epoch=True,
):
    set_seed(seed)
    train_loader, val_loader, _ = make_loaders(
        batch_size=int(params["batch_size"]),
        use_augmentation=use_augmentation,
        seed=seed,
    )
    dropout_rate = float(params["dropout"]) if use_dropout else 0.0
    model = FashionCNN(
        channels=int(params["channels"]),
        dropout=dropout_rate
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer_name = params.get("optimizer", "Adam")
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=float(params["lr"]))
    else:
        optimizer = torch.optim.SGD(
            model.parameters(), lr=float(params["lr"]), momentum=0.9
        )

    history = []
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * y.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate_model(model, val_loader, criterion)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        }
        history.append(row)

        if log_each_epoch:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_accuracy", train_acc, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        print(
            f"Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} "
            f"| train_acc={train_acc:.4f} | val_loss={val_loss:.4f} "
            f"| val_acc={val_acc:.4f}"
        )

    elapsed = time.time() - start
    return model, pd.DataFrame(history), elapsed

## 8. Baseline CNN

Baseline menjadi titik pembanding untuk eksperimen tuning dan ablation. Jalankan sel ini satu kali dan periksa bahwa run muncul di MLflow UI.

In [ ]:
BASELINE_PARAMS = {
    "lr": 0.001,
    "batch_size": 128,
    "channels": 16,
    "dropout": 0.25,
    "optimizer": "Adam",
}
BASELINE_EPOCHS = 3

with mlflow.start_run(run_name="baseline-cnn") as run:
    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_param("epochs", BASELINE_EPOCHS)
    mlflow.log_param("seed", 42)
    mlflow.log_param("dataset_hash", DATASET_HASH)
    mlflow.set_tag("stage", "baseline")
    mlflow.set_tag("dataset", "Fashion-MNIST")
    baseline_model, baseline_history, baseline_time = train_model(
        BASELINE_PARAMS, seed=42, epochs=BASELINE_EPOCHS
    )
    mlflow.log_metric("training_seconds", baseline_time)
    mlflow.log_metric("best_val_accuracy", float(baseline_history["val_accuracy"].max()))
    mlflow.log_metric("best_val_loss", float(baseline_history["val_loss"].min()))
    baseline_history.to_csv(ARTIFACT_DIR / "baseline_history.csv", index=False)
    mlflow.log_artifact(str(ARTIFACT_DIR / "baseline_history.csv"))
    input_example = torch.zeros(1, 1, 28, 28)

    mlflow.pytorch.log_model(
        baseline_model,
        name="baseline_model",
        input_example=input_example,
        serialization_format="pickle",
    )
    BASELINE_RUN_ID = run.info.run_id

print("Baseline run ID:", BASELINE_RUN_ID)
display(baseline_history)

Epoch 1/3 | train_loss=1.1594 | train_acc=0.5861 | val_loss=0.6776 | val_acc=0.7462
Epoch 2/3 | train_loss=0.6554 | train_acc=0.7574 | val_loss=0.5754 | val_acc=0.7921
Epoch 3/3 | train_loss=0.5691 | train_acc=0.7876 | val_loss=0.5083 | val_acc=0.8171


2026/10/09 05:49:25 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run baseline-cnn at: http://127.0.0.1:5000/#/experiments/1/runs/9c1973588b0342d6ae6fcae6d7e9b63f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Baseline run ID: 9c1973588b0342d6ae6fcae6d7e9b63f


,epoch,train_loss,train_accuracy,val_loss,val_accuracy
0,1,1.159428,0.586146,0.677638,0.746250
1,2,0.655385,0.757396,0.575381,0.792083
2,3,0.569067,0.787604,0.508319,0.817083


## 9. Hyperparameter tuning dengan Optuna

Bagian ini menjalankan 20 trial. Setiap trial dicatat sebagai nested run di bawah parent run. Untuk menghemat waktu, trial menggunakan tiga epoch dan subset data yang sudah ditentukan. Setelah pipeline tervalidasi, jumlah epoch atau ukuran data dapat ditingkatkan dan eksperimen diulang.

In [ ]:

N_TRIALS = 20
TUNING_EPOCHS = 3

def objective(trial):
    params = {
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
        "channels": trial.suggest_categorical("channels", [8, 16, 32]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "SGD"]),
    }

    with mlflow.start_run(
        run_name=f"trial-{trial.number:02d}",
        nested=True
    ):
        mlflow.log_params(params)
        mlflow.log_param("epochs", TUNING_EPOCHS)
        mlflow.log_param("seed", 42)
        mlflow.log_param("dataset_hash", DATASET_HASH)

        mlflow.set_tag("stage", "hyperparameter_tuning")
        mlflow.set_tag("dataset", "Fashion-MNIST")

        model, history, elapsed = train_model(
            params,
            seed=42,
            epochs=TUNING_EPOCHS,
            log_each_epoch=True
        )

        best_val_acc = float(history["val_accuracy"].max())
        best_val_loss = float(history["val_loss"].min())

        mlflow.log_metric("best_val_accuracy", best_val_acc)
        mlflow.log_metric("best_val_loss", best_val_loss)
        mlflow.log_metric("training_seconds", elapsed)

        mlflow.log_dict(
            history.to_dict(orient="records"),
            "training_history.json"
        )

        # Perbaikan penyimpanan model
        input_example = torch.zeros(1, 1, 28, 28)

        mlflow.pytorch.log_model(
            pytorch_model=model,
            name="trial_model",
            input_example=input_example,
            serialization_format="pickle",
        )

        trial.set_user_attr("best_val_loss", best_val_loss)

    return best_val_acc


with mlflow.start_run(run_name="optuna-sweep-20-trials"):
    mlflow.set_tag("stage", "tuning_parent")
    mlflow.log_param("n_trials", N_TRIALS)
    mlflow.log_param("dataset_hash", DATASET_HASH)

    study = optuna.create_study(
        direction="maximize",
        study_name="fashion_mnist_cnn"
    )

    study.optimize(objective, n_trials=N_TRIALS)

print("Jumlah trial selesai:", len(study.trials))
print("Best validation accuracy:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-10-09 05:56:36,198] A new study created in memory with name: fashion_mnist_cnn


Epoch 1/3 | train_loss=2.2968 | train_acc=0.0989 | val_loss=2.2893 | val_acc=0.1913
Epoch 2/3 | train_loss=2.2759 | train_acc=0.2016 | val_loss=2.2593 | val_acc=0.2471
Epoch 3/3 | train_loss=2.2232 | train_acc=0.3165 | val_loss=2.1712 | val_acc=0.3787


2026/10/09 05:57:07 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
[I 2026-10-09 05:57:40,195] Trial 0 finished with value: 0.37875 and parameters: {'lr': 0.0004665519119211146, 'batch_size': 64, 'channels': 8, 'dropout': 0.06250178464499312, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.37875.


🏃 View run trial-00 at: http://127.0.0.1:5000/#/experiments/1/runs/d9ee907534fa4596a7df9ae5cf66139b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Epoch 1/3 | train_loss=2.2869 | train_acc=0.1342 | val_loss=2.2575 | val_acc=0.2512
Epoch 2/3 | train_loss=2.1150 | train_acc=0.3262 | val_loss=1.7303 | val_acc=0.5317
Epoch 3/3 | train_loss=1.3075 | train_acc=0.5180 | val_loss=0.9425 | val_acc=0.6642


2026/10/09 05:58:04 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
[I 2026-10-09 05:58:25,089] Trial 1 finished with value: 0.6641666666666667 and parameters: {'lr': 0.0023271428847276765, 'batch_size': 128, 'channels': 8, 'dropout': 0.46691472402565704, 'optimizer': 'SGD'}. Best is trial 1 with value: 0.6641666666666667.


...

🏃 View run trial-17 at: http://127.0.0.1:5000/#/experiments/1/runs/ffa04f346a7e411d8e5f23a051dc8543
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Epoch 1/3 | train_loss=0.8193 | train_acc=0.6965 | val_loss=0.5069 | val_acc=0.8150
Epoch 2/3 | train_loss=0.4766 | train_acc=0.8255 | val_loss=0.4383 | val_acc=0.8467
Epoch 3/3 | train_loss=0.3949 | train_acc=0.8588 | val_loss=0.4037 | val_acc=0.8542


2026/10/09 06:22:07 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
[I 2026-10-09 06:22:27,179] Trial 18 finished with value: 0.8541666666666666 and parameters: {'lr': 0.0011876749014530519, 'batch_size': 64, 'channels': 32, 'dropout': 0.024978877366156955, 'optimizer': 'Adam'}. Best is trial 15 with value: 0.8620833333333333.


🏃 View run trial-18 at: http://127.0.0.1:5000/#/experiments/1/runs/e15903d554714706ad8896c1a83d4a3d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Epoch 1/3 | train_loss=2.2446 | train_acc=0.2357 | val_loss=2.0939 | val_acc=0.4562
Epoch 2/3 | train_loss=1.4506 | train_acc=0.5361 | val_loss=0.9620 | val_acc=0.6192
Epoch 3/3 | train_loss=0.8579 | train_acc=0.6716 | val_loss=0.7682 | val_acc=0.7079


2026/10/09 06:23:29 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
[I 2026-10-09 06:23:47,407] Trial 19 finished with value: 0.7079166666666666 and parameters: {'lr': 0.002791484550831913, 'batch_size': 128, 'channels': 32, 'dropout': 0.126268504921388, 'optimizer': 'SGD'}. Best is trial 15 with value: 0.8620833333333333.


🏃 View run trial-19 at: http://127.0.0.1:5000/#/experiments/1/runs/4fe6f52a3bba489aa9d727bd97eca608
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run optuna-sweep-20-trials at: http://127.0.0.1:5000/#/experiments/1/runs/39ab3102464148638a9650180605e9c0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Jumlah trial selesai: 20
Best validation accuracy: 0.8620833333333333
Best parameters: {'lr': 0.0028209365807357473, 'batch_size': 128, 'channels': 32, 'dropout': 0.023045806903528105, 'optimizer': 'Adam'}


## 10. Bandingkan run tuning melalui MLflow

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.stage = 'hyperparameter_tuning'",
    order_by=["metrics.best_val_accuracy DESC"],
    max_results=100,
)
display_cols = [
    c for c in [
        "run_id", "run_name", "params.lr", "params.batch_size",
        "params.channels", "params.dropout", "params.optimizer",
        "metrics.best_val_accuracy", "metrics.best_val_loss",
        "metrics.training_seconds"
    ] if c in runs_df.columns
]
display(runs_df[display_cols].head(20))

,run_id,params.lr,params.batch_size,params.channels,params.dropout,params.optimizer,metrics.best_val_accuracy,metrics.best_val_loss,metrics.training_seconds
0,64a8d1111a844e0dad3a3ebf914137ab,0.0028209365807357473,128,32,0.023045806903528105,Adam,0.862083,0.398204,65.986608
1,4d8ec831058348948540f276c4111cf7,0.002017955249756148,128,32,0.01698533952064583,Adam,0.859167,0.397820,60.859416
2,91040f97d0eb44f49bdb47d712c9df96,0.002060813984525167,128,32,0.184849961051097,Adam,0.854583,0.404904,59.437343
3,e15903d554714706ad8896c1a83d4a3d,0.0011876749014530519,64,32,0.024978877366156955,Adam,0.854167,0.403718,66.755845
4,9146f284d68641a79109e5d7a92b8454,0.0021109937284009878,128,32,0.033768323698242064,Adam,0.852500,0.415991,80.933511
5,191e24219de242a7850f71b7b12915b1,0.0009446232355133788,128,32,0.3142874231328159,Adam,0.835000,0.450434,57.463208
6,ebf28d57ced74d5ba4f1343a6a3983bf,0.0011630503485973629,128,32,0.4318740661141246,Adam,0.833750,0.447875,80.112477
7,a33b941ea6ae462eaa47ce6b3a528306,0.00026513823931028437,64,32,0.1168201302899578,Adam,0.818750,0.513567,80.656411
8,db774b610efe4b179fa83e559754fdb0,0.0010639349877943397,128,32,0.4892689848542572,Adam,0.810000,0.476448,92.785058
9,1a48607debb846ad89317bd18e427eb9,0.0003490050942875852,128,32,0.13003300838984638,Adam,0.799583,0.535711,64.436976


## 11. Uji konfigurasi terbaik dengan tiga seed

Bagian ini menguji kestabilan konfigurasi terbaik. Nilai mean dan standard deviation dihitung dari tiga hasil validasi. Untuk laporan, catat nilai setiap seed sekaligus ringkasannya.

In [ ]:
BEST_PARAMS = dict(study.best_params)
SEEDS = [42, 123, 2026]
MULTI_SEED_EPOCHS = 3
seed_results = []

for seed in SEEDS:
    with mlflow.start_run(run_name=f"best-config-seed-{seed}"):
        mlflow.log_params(BEST_PARAMS)
        mlflow.log_param("seed", seed)
        mlflow.log_param("epochs", MULTI_SEED_EPOCHS)
        mlflow.log_param("dataset_hash", DATASET_HASH)
        mlflow.set_tag("stage", "multi_seed_evaluation")
        model, history, elapsed = train_model(
            BEST_PARAMS, seed=seed, epochs=MULTI_SEED_EPOCHS
        )
        best_acc = float(history["val_accuracy"].max())
        best_loss = float(history["val_loss"].min())
        mlflow.log_metric("best_val_accuracy", best_acc)
        mlflow.log_metric("best_val_loss", best_loss)
        mlflow.log_metric("training_seconds", elapsed)
        input_example = torch.zeros(1, 1, 28, 28)

        mlflow.pytorch.log_model(
            pytorch_model=model,
            name=f"model_seed_{seed}",
            input_example=input_example,
            serialization_format="pickle",
        )
        seed_results.append({
            "seed": seed,
            "best_val_accuracy": best_acc,
            "best_val_loss": best_loss,
            "training_seconds": elapsed,
        })

seed_results_df = pd.DataFrame(seed_results)
display(seed_results_df)
print("Mean validation accuracy:", seed_results_df["best_val_accuracy"].mean())
print("Std validation accuracy:", seed_results_df["best_val_accuracy"].std(ddof=1))
print("Mean validation loss:", seed_results_df["best_val_loss"].mean())
print("Std validation loss:", seed_results_df["best_val_loss"].std(ddof=1))
seed_results_df.to_csv(ARTIFACT_DIR / "multi_seed_results.csv", index=False)

Epoch 1/3 | train_loss=0.8229 | train_acc=0.6956 | val_loss=0.4881 | val_acc=0.8250
Epoch 2/3 | train_loss=0.4470 | train_acc=0.8386 | val_loss=0.4013 | val_acc=0.8588
Epoch 3/3 | train_loss=0.3654 | train_acc=0.8703 | val_loss=0.3982 | val_acc=0.8621


2026/10/09 06:43:30 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run best-config-seed-42 at: http://127.0.0.1:5000/#/experiments/1/runs/d32ec2d0f85646b5b90ab469bcd297f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Epoch 1/3 | train_loss=0.7533 | train_acc=0.7232 | val_loss=0.4835 | val_acc=0.8158
Epoch 2/3 | train_loss=0.4291 | train_acc=0.8472 | val_loss=0.4078 | val_acc=0.8600
Epoch 3/3 | train_loss=0.3570 | train_acc=0.8691 | val_loss=0.3805 | val_acc=0.8671


2026/10/09 06:45:04 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run best-config-seed-123 at: http://127.0.0.1:5000/#/experiments/1/runs/b956d69da7734bf9877b2e6ad55fa22d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Epoch 1/3 | train_loss=0.8252 | train_acc=0.7021 | val_loss=0.5088 | val_acc=0.8179
Epoch 2/3 | train_loss=0.4578 | train_acc=0.8356 | val_loss=0.4259 | val_acc=0.8525
Epoch 3/3 | train_loss=0.3930 | train_acc=0.8597 | val_loss=0.3710 | val_acc=0.8654


2026/10/09 06:46:21 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run best-config-seed-2026 at: http://127.0.0.1:5000/#/experiments/1/runs/c72a59aabb3b45379acee74bb1b665ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,seed,best_val_accuracy,best_val_loss,training_seconds
0,42,0.862083,0.398204,67.955158
1,123,0.867083,0.380525,59.879678
2,2026,0.865417,0.370966,58.721071


Mean validation accuracy: 0.864861111111111
Std validation accuracy: 0.002545875386086576
Mean validation loss: 0.3832317037052579
Std validation loss: 0.01381944945037108


## 12. Ablation study 1: augmentasi

Bandingkan pelatihan dengan augmentasi dan tanpa augmentasi. Parameter model, seed, jumlah epoch, dan pembagian data dipertahankan sama.

In [33]:
ABLATION_EPOCHS = 3
ablation_results = []

for use_aug in [False, True]:
    label = "with_augmentation" if use_aug else "without_augmentation"
    params = dict(BEST_PARAMS)
    with mlflow.start_run(run_name=f"ablation-{label}"):
        mlflow.log_params(params)
        mlflow.log_param("seed", 42)
        mlflow.log_param("epochs", ABLATION_EPOCHS)
        mlflow.log_param("use_augmentation", use_aug)
        mlflow.log_param("dataset_hash", DATASET_HASH)
        mlflow.set_tag("stage", "ablation")

        model, history, elapsed = train_model(
            params, seed=42, epochs=ABLATION_EPOCHS,
            use_augmentation=use_aug
        )
        result = {
            "ablation": label,
            "best_val_accuracy": float(history["val_accuracy"].max()),
            "best_val_loss": float(history["val_loss"].min()),
            "training_seconds": elapsed,
        }
        mlflow.log_metrics({
            "best_val_accuracy": result["best_val_accuracy"],
            "best_val_loss": result["best_val_loss"],
            "training_seconds": elapsed,
        })
        mlflow.pytorch.log_model(
            model,
            name=f"model_{label}",          # <-- diperbaiki
            input_example=torch.zeros(1, 1, 28, 28),
            serialization_format="pickle",
        )
        ablation_results.append(result)

ablation_aug_df = pd.DataFrame(ablation_results)
display(ablation_aug_df)

Epoch 1/3 | train_loss=0.8229 | train_acc=0.6956 | val_loss=0.4881 | val_acc=0.8250
Epoch 2/3 | train_loss=0.4470 | train_acc=0.8386 | val_loss=0.4013 | val_acc=0.8588
Epoch 3/3 | train_loss=0.3654 | train_acc=0.8703 | val_loss=0.3982 | val_acc=0.8621


2026/10/09 12:14:50 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run ablation-without_augmentation at: http://127.0.0.1:5000/#/experiments/1/runs/48a3e7f40f1f4146996c23725a6a120b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Epoch 1/3 | train_loss=0.9050 | train_acc=0.6702 | val_loss=0.5521 | val_acc=0.8058
Epoch 2/3 | train_loss=0.5330 | train_acc=0.8035 | val_loss=0.4702 | val_acc=0.8296
Epoch 3/3 | train_loss=0.4475 | train_acc=0.8368 | val_loss=0.4332 | val_acc=0.8471


2026/10/09 12:17:11 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run ablation-with_augmentation at: http://127.0.0.1:5000/#/experiments/1/runs/5ab9d47fe915485fbf12cb5d9a49bcda
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,ablation,best_val_accuracy,best_val_loss,training_seconds
0,without_augmentation,0.862083,0.398204,73.892320
1,with_augmentation,0.847083,0.433205,89.992871


## 13. Ablation study 2: dropout

In [35]:
dropout_results = []

for use_dropout in [False, True]:
    label = "with_dropout" if use_dropout else "without_dropout"
    params = dict(BEST_PARAMS)
    with mlflow.start_run(run_name=f"ablation-{label}"):
        mlflow.log_params(params)
        mlflow.log_param("seed", 42)
        mlflow.log_param("epochs", ABLATION_EPOCHS)
        mlflow.log_param("use_dropout", use_dropout)
        mlflow.log_param("dataset_hash", DATASET_HASH)
        mlflow.set_tag("stage", "ablation")
        model, history, elapsed = train_model(
            params, seed=42, epochs=ABLATION_EPOCHS,
            use_augmentation=False, use_dropout=use_dropout
        )
        result = {
            "ablation": label,
            "best_val_accuracy": float(history["val_accuracy"].max()),
            "best_val_loss": float(history["val_loss"].min()),
            "training_seconds": elapsed,
        }
        mlflow.log_metrics({
            "best_val_accuracy": result["best_val_accuracy"],
            "best_val_loss": result["best_val_loss"],
            "training_seconds": elapsed,
        })
        mlflow.pytorch.log_model(
                    model,
                    name=f"model_{label}",          # <-- diperbaiki
                    input_example=torch.zeros(1, 1, 28, 28),
                    serialization_format="pickle",
                )
        dropout_results.append(result)

ablation_dropout_df = pd.DataFrame(dropout_results)
display(ablation_dropout_df)

Epoch 1/3 | train_loss=0.8054 | train_acc=0.7060 | val_loss=0.4840 | val_acc=0.8296
Epoch 2/3 | train_loss=0.4375 | train_acc=0.8441 | val_loss=0.4045 | val_acc=0.8592
Epoch 3/3 | train_loss=0.3591 | train_acc=0.8729 | val_loss=0.4039 | val_acc=0.8596


2026/10/09 12:27:03 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run ablation-without_dropout at: http://127.0.0.1:5000/#/experiments/1/runs/0c863b436aa74bf08a2e9c387e305b36
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Epoch 1/3 | train_loss=0.8229 | train_acc=0.6956 | val_loss=0.4881 | val_acc=0.8250
Epoch 2/3 | train_loss=0.4470 | train_acc=0.8386 | val_loss=0.4013 | val_acc=0.8588
Epoch 3/3 | train_loss=0.3654 | train_acc=0.8703 | val_loss=0.3982 | val_acc=0.8621


2026/10/09 12:28:39 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run ablation-with_dropout at: http://127.0.0.1:5000/#/experiments/1/runs/207d0399ac4848f387abc377ea375060
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,ablation,best_val_accuracy,best_val_loss,training_seconds
0,without_dropout,0.859583,0.403907,69.479120
1,with_dropout,0.862083,0.398204,71.698657


## 14. Latih model final dan evaluasi test set satu kali

Setelah konfigurasi dipilih berdasarkan validasi, model final dilatih menggunakan konfigurasi terbaik. Test set digunakan di tahap ini untuk melaporkan performa final, bukan untuk memilih hyperparameter.

In [37]:
FINAL_SEED = 42
FINAL_EPOCHS = 5

with mlflow.start_run(run_name="final-model-test-evaluation") as final_run:
    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("seed", FINAL_SEED)
    mlflow.log_param("epochs", FINAL_EPOCHS)
    mlflow.log_param("dataset_hash", DATASET_HASH)
    mlflow.set_tag("stage", "final_evaluation")

    final_model, final_history, final_time = train_model(
        BEST_PARAMS, seed=FINAL_SEED, epochs=FINAL_EPOCHS
    )
    _, _, final_test_loader = make_loaders(
        batch_size=int(BEST_PARAMS["batch_size"]), seed=FINAL_SEED
    )
    test_loss, test_accuracy = evaluate_model(
        final_model, final_test_loader, nn.CrossEntropyLoss()
    )
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("training_seconds", final_time)

    mlflow.pytorch.log_model(
        final_model,
        name="final_model",
        input_example=torch.zeros(1, 1, 28, 28),
        serialization_format="pickle",
    )
    FINAL_RUN_ID = final_run.info.run_id

print("Final run ID:", FINAL_RUN_ID)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

Epoch 1/5 | train_loss=0.8229 | train_acc=0.6956 | val_loss=0.4881 | val_acc=0.8250
Epoch 2/5 | train_loss=0.4470 | train_acc=0.8386 | val_loss=0.4013 | val_acc=0.8588
Epoch 3/5 | train_loss=0.3654 | train_acc=0.8703 | val_loss=0.3982 | val_acc=0.8621
Epoch 4/5 | train_loss=0.3156 | train_acc=0.8848 | val_loss=0.3615 | val_acc=0.8758
Epoch 5/5 | train_loss=0.2803 | train_acc=0.8958 | val_loss=0.3654 | val_acc=0.8704


2026/10/09 12:36:39 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run final-model-test-evaluation at: http://127.0.0.1:5000/#/experiments/1/runs/b5ea1a511d824a4f8271b12468fce903
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Final run ID: b5ea1a511d824a4f8271b12468fce903
Test loss: 0.3767
Test accuracy: 0.8650


## 15. Daftarkan model pada MLflow Model Registry

Model didaftarkan dari run final. MLflow server harus menggunakan backend database yang mendukung Model Registry, seperti SQLite pada perintah server di bagian persiapan. Jika registrasi gagal, simpan error lengkap dan periksa log server.

In [38]:
model_uri = f"runs:/{FINAL_RUN_ID}/final_model"
registered = mlflow.register_model(model_uri=model_uri, name=REGISTERED_MODEL_NAME)
print("Registered model name:", registered.name)
print("Registered model version:", registered.version)

Successfully registered model 'fashion-mnist-cnn'.
2026/10/09 12:37:16 WARNING mlflow.tracking._model_registry.fluent: Run with id b5ea1a511d824a4f8271b12468fce903 has no artifacts at artifact path 'final_model', registering model based on models:/m-3cc80e40245448a0a34849394ea78a16 instead
2026/10/09 12:37:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fashion-mnist-cnn, version 1
Created version '1' of model 'fashion-mnist-cnn'.


Registered model name: fashion-mnist-cnn
Registered model version: 1


## 16. Tetapkan alias champion

In [39]:
client = mlflow.tracking.MlflowClient()
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="champion",
    version=registered.version,
)
print(f"Alias champion ditetapkan ke versi {registered.version}")

Alias champion ditetapkan ke versi 1


## 17. Muat model melalui alias champion dan uji prediksi

Tahap ini membuktikan bahwa model dapat dimuat melalui nama terdaftar dan alias, bukan hanya melalui variabel model yang masih berada di memori.

In [40]:
champion_uri = f"models:/{REGISTERED_MODEL_NAME}@champion"
champion_model = mlflow.pytorch.load_model(champion_uri)
champion_model = champion_model.to(DEVICE)
champion_model.eval()

sample_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
sample_images, sample_labels = next(iter(sample_loader))
with torch.no_grad():
    logits = champion_model(sample_images.to(DEVICE))
    predicted_labels = logits.argmax(dim=1).cpu().numpy()

prediction_table = pd.DataFrame({
    "actual_label": sample_labels.numpy(),
    "actual_class": [CLASS_NAMES[int(i)] for i in sample_labels.numpy()],
    "predicted_label": predicted_labels,
    "predicted_class": [CLASS_NAMES[int(i)] for i in predicted_labels],
})
display(prediction_table)

,actual_label,actual_class,predicted_label,predicted_class
0,9,Ankle boot,9,Ankle boot
1,2,Pullover,2,Pullover
2,1,Trouser,1,Trouser
3,1,Trouser,1,Trouser
4,6,Shirt,6,Shirt
5,1,Trouser,1,Trouser
6,4,Coat,4,Coat
7,6,Shirt,6,Shirt


## 18. Simpan ringkasan hasil untuk laporan
Sel ini menggabungkan ringkasan yang sudah dihasilkan notebook ke file JSON. File tersebut dapat digunakan saat memperbarui laporan DOCX.

In [43]:
# 1) Baseline dan multi-seed: dibaca dari CSV yang disimpan notebook sebelumnya
for f in ["baseline_history.csv", "multi_seed_results.csv"]:
    assert (ARTIFACT_DIR / f).exists(), f"File tidak ditemukan: {ARTIFACT_DIR / f}"

baseline_history = pd.read_csv(ARTIFACT_DIR / "baseline_history.csv")
seed_results_df = pd.read_csv(ARTIFACT_DIR / "multi_seed_results.csv")

# 2) Hasil tuning: study Optuna hanya ada di memori, jadi dihitung dari run MLflow
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
tuning_runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.stage = 'hyperparameter_tuning' AND attributes.status = 'FINISHED'",
)
n_trials = len(tuning_runs)
best_val = float(tuning_runs["metrics.best_val_accuracy"].max())

# 3) Sanity check sebelum menyimpan
print("Jumlah trial tuning :", n_trials, "(harus 20)")
print("Best tuning val acc :", round(best_val, 4), "(sebelumnya 0.8621)")
print("Baseline best val   :", round(float(baseline_history['val_accuracy'].max()), 4), "(sebelumnya 0.8171)")
print("Seed mean / std     :", round(float(seed_results_df['best_val_accuracy'].mean()), 4),
      "/", round(float(seed_results_df['best_val_accuracy'].std(ddof=1)), 4), "(sebelumnya 0.8649 / 0.0025)")
print("Baris ablation      :", len(ablation_aug_df), "dan", len(ablation_dropout_df), "(masing-masing harus 2)")

# 4) Simpan ringkasan
summary = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset": "Fashion-MNIST",
    "dataset_hash": DATASET_HASH,
    "baseline_best_val_accuracy": float(baseline_history["val_accuracy"].max()),
    "tuning_trials_completed": int(n_trials),
    "best_tuning_val_accuracy": best_val,
    "best_params": BEST_PARAMS,
    "multi_seed_mean_val_accuracy": float(seed_results_df["best_val_accuracy"].mean()),
    "multi_seed_std_val_accuracy": float(seed_results_df["best_val_accuracy"].std(ddof=1)),
    "augmentation_ablation": ablation_aug_df.to_dict(orient="records"),
    "dropout_ablation": ablation_dropout_df.to_dict(orient="records"),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "registered_model_name": REGISTERED_MODEL_NAME,
    "registered_model_version": str(registered.version),
    "model_alias": "champion",
    "final_run_id": FINAL_RUN_ID,
}
with open(ARTIFACT_DIR / "experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

Jumlah trial tuning : 20 (harus 20)
Best tuning val acc : 0.8621 (sebelumnya 0.8621)
Baseline best val   : 0.8171 (sebelumnya 0.8171)
Seed mean / std     : 0.8649 / 0.0025 (sebelumnya 0.8649 / 0.0025)
Baris ablation      : 2 dan 2 (masing-masing harus 2)
{
  "experiment_name": "fashion-mnist-cnn-mlops-week6",
  "dataset": "Fashion-MNIST",
  "dataset_hash": "077b4755137a74af2a91fab67a4119b20dc1a4414eceff32227b0665a60cd1b9",
  "baseline_best_val_accuracy": 0.8170833333333334,
  "tuning_trials_completed": 20,
  "best_tuning_val_accuracy": 0.8620833333333333,
  "best_params": {
    "lr": 0.0028209365807357473,
    "batch_size": 128,
    "channels": 32,
    "dropout": 0.023045806903528105,
    "optimizer": "Adam"
  },
  "multi_seed_mean_val_accuracy": 0.864861111111111,
  "multi_seed_std_val_accuracy": 0.002545875386086576,
  "augmentation_ablation": [
    {
      "ablation": "without_augmentation",
      "best_val_accuracy": 0.8620833333333333,
      "best_val_loss": 0.39820418318112694,
 

## 19. Checklist akhir

- [ ] MLflow Tracking Server berjalan dan UI dapat dibuka.
- [ ] Baseline CNN tercatat, termasuk metrik per epoch dan model.
- [ ] Minimal 20 trial tuning selesai dan terlihat sebagai nested runs.
- [ ] Konfigurasi terbaik dijalankan dengan tiga seed.
- [ ] Mean dan standard deviation dicatat dalam laporan.
- [ ] Ablation augmentasi dan dropout selesai.
- [ ] Versi dataset dilacak oleh DVC dan hash dicatat pada setiap run.
- [ ] Model final terdaftar pada Model Registry.
- [ ] Alias `champion` berhasil ditetapkan.
- [ ] Model berhasil dimuat melalui `models:/fashion-mnist-cnn@champion`.
- [ ] Prediksi contoh berhasil dilakukan.
- [ ] Screenshot MLflow UI compare view disimpan.
- [ ] Repository Git, README, dan laporan DOCX diperbarui dengan hasil aktual.

### Catatan interpretasi

Jangan menyimpulkan bahwa satu konfigurasi lebih baik hanya karena satu run terlihat bagus. Bandingkan metrik validasi dengan prosedur yang konsisten, lihat variasi tiga seed, dan jelaskan keterbatasan penggunaan subset data serta jumlah epoch yang relatif kecil.